In [ ]:
import ee
import geemap
import os
import shutil
import zipfile
from datetime import datetime, timedelta
# Authenticate with the service account
service_account_key = '******'
credentials = ee.ServiceAccountCredentials('*******', service_account_key)
ee.Initialize(credentials)

In [ ]:
# Define the CHIRPS dataset
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# Define the list of countries
region_countries = ["Tunisia"]
# Use the USDOS/LSIB_SIMPLE/2017 dataset for country boundaries
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
# Directory in Kaggle to save .tif files
save_dir = '/kaggle/working/SPI_Images'
os.makedirs(save_dir, exist_ok=True)
# Function to sanitize country names for safe filenames
def sanitize_country_name(country_name):
    return country_name.replace("'", "").replace(" ", "_")
# Function to get climatological monthly statistics (1981-2010 standard period)
def get_monthly_climatology(month):
    """Calculate mean and stdDev of monthly precipitation sums for specific month across 30 years"""
    years = ee.List.sequence(1981, 2025)

    def process_year(year):
        year = ee.Number(year)
        monthly_sum = chirps \
            .filter(ee.Filter.calendarRange(year, year, 'year')) \
            .filter(ee.Filter.calendarRange(month, month, 'month')) \
            .select('precipitation') \
            .sum()
        return monthly_sum.set('year', year)

    yearly_sums = ee.ImageCollection.fromImages(years.map(process_year))
    mean = yearly_sums.mean().rename('mean')
    std_dev = yearly_sums.reduce(ee.Reducer.stdDev()).rename('stdDev')

    return mean.addBands(std_dev)

# Function to calculate SPI for a country, year, and month
def calculate_spi(country_geometry, year, month):
    # Get climatological statistics for this month
    climatology = get_monthly_climatology(month)
    mean = climatology.select('mean')
    std_dev = climatology.select('stdDev')

    # Current month precipitation sum
    current_month_precipitation = chirps \
        .filter(ee.Filter.calendarRange(year, year, 'year')) \
        .filter(ee.Filter.calendarRange(month, month, 'month')) \
        .select('precipitation') \
        .sum() \
        .rename('current_month_sum')
    # Calculate SPI: (current - mean) / stdDev
    spi = current_month_precipitation.subtract(mean).divide(std_dev).rename('SPI')
    spi = spi.max(-3).min(3)
    return spi.clip(country_geometry)

# Process SPI for each country
for country_name in region_countries:
    print(f"Processing SPI for {country_name}...")
    # Sanitize country name for file paths
    safe_country_name = sanitize_country_name(country_name)
    # Create a separate folder for the country
    country_save_dir = os.path.join(save_dir, safe_country_name)
    os.makedirs(country_save_dir, exist_ok=True)
    # Get the geometry for the country (server-side)
    country = countries.filter(ee.Filter.eq('country_na', country_name)).first()
    geometry = country.geometry()
    # Get region bounds for export (client-side safe)
    region_info = geometry.bounds().getInfo()
    export_region = ee.Geometry.Polygon(region_info['coordinates'])

    # Process each year and month
    for year in range(2000, 2025):
        for month in range(1, 13):
            print(f"  Processing {year}-{month:02d}...")
            spi_image = calculate_spi(geometry, year, month)
            description = f"{safe_country_name}_SPI_{year}_{month:02d}"
            file_path = os.path.join(country_save_dir, f"{description}.tif")
            # Export SPI image
            geemap.ee_export_image(
                spi_image,
                filename=file_path,
                scale=1000,  # ~5km matching CHIRPS resolution
                region=export_region,
                file_per_band=False
            )
            print(f"    Exported: {description}")
    # Compress the folder into a ZIP file
    zip_file_path = f"/kaggle/working/{safe_country_name}_SPI.zip"
    with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(country_save_dir):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, country_save_dir)
                zipf.write(file_path, arcname)
    # Delete the country folder to save space
    shutil.rmtree(country_save_dir)
    print(f"SPI data for {country_name} saved and compressed as {zip_file_path}.")

In [ ]:

# LST

# Define the CHIRPS dataset
chirps = ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY")
# Define the list of countries for West Africa
region_countries = [
    "Tunisia"
]
# Use the USDOS/LSIB_SIMPLE/2017 dataset for country boundaries
countries = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
# Define the date range for the dataset
start_year = 2000
end_year = 2026

# MODIS dataset
dataset = ee.ImageCollection('MODIS/061/MOD11A1')

# Output directory
output_base_dir = '/kaggle/working/EarthEngineExports'

# Loop through each country
for country in region_countries :
    print(f"Processing {country}...")

    # Adjust folder name for "Cote d'Ivoire"
    folder_name = country.replace(" ", "_")
    output_dir = os.path.join(output_base_dir, folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # Filter region by country name
    region = countries.filter(ee.Filter.eq('country_na', country))

    # Process each year and month
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            start_date = f"{year:04d}-{month:02d}-01"
            end_date = ee.Date(start_date).advance(1, 'month').format('YYYY-MM-dd').getInfo()

            # Calculate monthly mean
            monthly_mean = dataset.filterDate(start_date, end_date) \
                                  .select('LST_Day_1km') \
                                  .mean() \
                                  .clip(region)

            # Apply conversion: Kelvin to Celsius
            converted_mean = monthly_mean.multiply(0.02).subtract(273.15)

            # 🔹 FIX: Mask NoData pixels outside the country
            converted_mean = converted_mean.updateMask(converted_mean)

            # Export image to file
            file_name = f"LST_MonthlyMean_{year}_{month:02d}.tif"
            file_path = os.path.join(output_dir, file_name)

            geemap.ee_export_image(
                converted_mean,
                filename=file_path,
                scale=1000,
                region=region.geometry(),
                crs='EPSG:4326'
            )

    # Compress and remove original folder
    zip_path = f"{output_dir}_LST.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for root, _, files in os.walk(output_dir):
            for file in files:
                zf.write(os.path.join(root, file), os.path.relpath(os.path.join(root, file), output_dir))
    print(f"Compressed {country} folder into {zip_path}")

    # Remove unzipped folder
    for root, _, files in os.walk(output_dir):
        for file in files:
            os.remove(os.path.join(root, file))
    os.rmdir(output_dir)

print("Processing complete!")

In [ ]:
# NDVI
# Define the date range for the MODIS NDVI dataset
begin_date = '2000-01-01'
end_date = '2026-12-31'
date_format = "%Y-%m-%d"
region_countries = ["Tunisia"]
# Directory in Kaggle to save .tif files
save_dir = '/kaggle/working/NDVI_Images'
os.makedirs(save_dir, exist_ok=True)

# Function to sanitize country names for safe filenames
def sanitize_country_name(country_name):
    return country_name.replace("'", "").replace(" ", "_")

# Function to download image as GeoTIFF with WGS84 projection
def download_image(image, description, region, save_dir, scale=1000, crs="EPSG:4326"):
    path = os.path.join(save_dir, f"{description}.tif")
    geemap.ee_export_image(image, filename=path, scale=scale, region=region, crs=crs, file_per_band=False)
    print(f"Image saved as {path}")

# Function to zip and delete folder
def zip_and_delete_folder(folder_path):
    zip_path = folder_path + ".zip"
    shutil.make_archive(folder_path, 'zip', folder_path)
    shutil.rmtree(folder_path)  # Delete the original folder
    print(f"Zipped and deleted folder: {zip_path}")

# Define start and end dates for filtering
start_date = datetime.strptime(begin_date, date_format)
end_date = datetime.strptime(end_date, date_format)
# Loop over each country to download images separately
for country_name in region_countries:
    # Sanitize the country name for filename
    safe_country_name = sanitize_country_name(country_name)

    # Create a separate folder for each country inside the main directory
    country_save_dir = os.path.join(save_dir, safe_country_name)
    os.makedirs(country_save_dir, exist_ok=True)

    # Filter the specific country
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    geometry = country.geometry()

    # Loop through each month between 2000 and 2023
    current_date = start_date
    while current_date <= end_date:
        # Move to the next month
        next_month = current_date.replace(day=28) + timedelta(days=4)
        next_month = next_month.replace(day=1)  # Set to the first day of the next month

        # Filter the MODIS image collection for the current month and country
        dataset = ee.ImageCollection('MODIS/061/MOD13A3') \
            .filterDate(current_date.strftime(date_format), next_month.strftime(date_format)) \
            .map(lambda image: image.clip(geometry))

        # Get the NDVI image (one per month)
        ndvi_image = dataset.select('NDVI').first().multiply(0.0001)  # Scale NDVI values

        # Save the image to the country's folder with sanitized country and date-specific naming
        description = f'NDVI_{safe_country_name}_{current_date.strftime("%Y_%m")}'
        try:
            download_image(ndvi_image, description, geometry, country_save_dir, scale=1000, crs="EPSG:4326")
        except Exception as e:
            print(f"Error downloading {description}: {e}")

        # Move to the next month
        current_date = next_month

    # Zip the country folder and delete the unzipped files
    zip_and_delete_folder(country_save_dir)

print("All images downloaded, zipped, and cleaned up successfully.")

In [ ]:
# Soil Moisture
# Define the date range for GLDAS data (soil moisture)
begin_date = '2000-01-01'
end_date = '2026-12-31'
date_format = "%Y-%m-%d"

# Directory in Kaggle to save .tif files
save_dir = '/kaggle/working/GLDAS_Soil_Moisture'
os.makedirs(save_dir, exist_ok=True)

# Function to sanitize country names for safe filenames
def sanitize_country_name(country_name):
    return country_name.replace("'", "").replace(" ", "_").replace(",", "")

# Function to download image as GeoTIFF to Kaggle
def download_image(image, description, region, save_dir, scale=1000):
    path = os.path.join(save_dir, f"{description}.tif")
    geemap.ee_export_image(image, filename=path, scale=scale, region=region, file_per_band=False)
    print(f"Image saved as {path}")

# Function to zip a folder and delete the original folder
def zip_and_delete_folder(folder_path):
    zip_path = f"{folder_path}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, folder_path)
                zipf.write(file_path, arcname)
    print(f"Folder zipped as {zip_path}")
    # Delete the original folder
    for root, dirs, files in os.walk(folder_path, topdown=False):
        for file in files:
            os.remove(os.path.join(root, file))
        for dir in dirs:
            os.rmdir(os.path.join(root, dir))
    os.rmdir(folder_path)
    print(f"Deleted folder: {folder_path}")

# Loop over each country to download images separately
for country_name in region_countries:
    # Sanitize the country name for filename
    safe_country_name = sanitize_country_name(country_name)

    # Create a subdirectory for the country
    country_dir = os.path.join(save_dir, safe_country_name)
    os.makedirs(country_dir, exist_ok=True)

    # Filter the specific country
    country = countries.filter(ee.Filter.eq('country_na', country_name))
    geometry = country.geometry()

    # Define start and end dates
    current_date = datetime.strptime(begin_date, date_format)
    end_date_obj = datetime.strptime(end_date, date_format)

    # Loop through each month within the date range
    while current_date <= end_date_obj:
        # Calculate the end of the current month
        next_month = current_date.replace(day=28) + timedelta(days=4)  # Move to next month
        end_of_month = next_month - timedelta(days=next_month.day)

        # Select the GLDAS soil moisture images for the current month
        dataset = ee.ImageCollection('NASA/GLDAS/V021/NOAH/G025/T3H') \
            .filterDate(current_date.strftime(date_format), end_of_month.strftime(date_format)) \
            .map(lambda image: image.clip(geometry).mask(ee.Image().paint(geometry, 1)))

        # Calculate the monthly average soil moisture
        monthly_soil_moisture = dataset.select('SoilMoi0_10cm_inst').mean()

        # Check if image is available for the specific month
        if monthly_soil_moisture:
            # Save the image to the country's directory with sanitized country and date-specific naming
            description = f'GLDAS_SoilMoisture_{safe_country_name}_{current_date.strftime("%Y_%m")}'
            try:
                download_image(monthly_soil_moisture, description, geometry, country_dir, scale=1000)  # Scale set to 1km for GLDAS
            except Exception as e:
                print(f"Error downloading {description}: {e}")

        # Move to the next month
        current_date = end_of_month + timedelta(days=1)

    # Zip the country's folder and delete the original folder
    zip_and_delete_folder(country_dir)

print("All monthly soil moisture images downloaded, zipped, and folders deleted successfully.")

In [ ]:
#SPEI
save_dir = "/kaggle/working/Tunisia_SPEI"
os.makedirs(save_dir, exist_ok=True)
zip_output = "/kaggle/working/Tunisia_SPEI.zip"
scale_m = 1000
crs_out = "EPSG:4326"
tunisia = (
    ee.FeatureCollection("FAO/GAUL/2015/level0")
    .filter(ee.Filter.eq("ADM0_NAME", "Tunisia"))
)
tun_geom = tunisia.geometry()
era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY")
    .filterDate("2000-01-01", "2025-12-31")
)
P = era5.select("total_precipitation").map(
    lambda img: img.multiply(1000)
    .rename("P")
    .clip(tun_geom)
    .copyProperties(img, ["system:time_start"])
)

PET = era5.select("potential_evaporation").map(
    lambda img: img.multiply(1000).abs()
    .rename("PET")
    .clip(tun_geom)
    .copyProperties(img, ["system:time_start"])
)
joined = ee.Join.inner().apply(
    P,
    PET,
    ee.Filter.equals(
        leftField="system:time_start",
        rightField="system:time_start"
    )
)

D = ee.ImageCollection(
    joined.map(
        lambda f: ee.Image(
            ee.Image(ee.Feature(f).get("primary"))
            .subtract(ee.Image(ee.Feature(f).get("secondary")))
            .rename("D")
            .set(
                "system:time_start",
                ee.Image(ee.Feature(f).get("primary")).get("system:time_start")
            )
        )
    )
)
meanD = D.reduce(ee.Reducer.mean())
stdD  = D.reduce(ee.Reducer.stdDev())

Z = D.map(
    lambda img: img.subtract(meanD)
    .divide(stdD)
    .rename("SPEI1")
    .set("system:time_start", img.get("system:time_start"))
)
for year in range(2000, 2025):
    for month in range(1, 13):
        start = ee.Date.fromYMD(year, month, 1)
        end   = start.advance(1, "month")
        img = Z.filterDate(start, end).first()
        if img is None:
            print(f"Skipping {year}-{month:02d} (no data)")
            continue
        print(f"Exporting {year}-{month:02d}")
        filename = f"Tunisia_SPEI_{year}_{month:02d}.tif"
        filepath = os.path.join(save_dir, filename)
        geemap.ee_export_image(
            img.clip(tun_geom),
            filename=filepath,
            scale=scale_m,
            region=tun_geom,
            crs=crs_out,
            file_per_band=False
        )
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(save_dir):
        for f in files:
            full_path = os.path.join(root, f)
            zipf.write(full_path, arcname=f)

shutil.rmtree(save_dir)

print("\n All months exported and zipped:")
print(zip_output)